# Epic 3: TSP Solver Implementation Demo

This notebook demonstrates the implementation of **US-301** (Nearest Neighbor Heuristic) and **US-302** (2-Opt Local Search) for solving the Traveling Salesman Problem on Singapore's MRT/LRT network.

## Table of Contents
1. [Setup and Data Loading](#setup)
2. [US-301: Nearest Neighbor Heuristic](#us-301)
3. [US-302: 2-Opt Local Search](#us-302)
4. [Combined Workflow: NN + 2-Opt](#combined)
5. [Performance Analysis](#performance)
6. [Conclusions](#conclusions)

<a id='setup'></a>
## 1. Setup and Data Loading

First, let's import the necessary modules and load the Singapore MRT/LRT network graph.

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '..')

import networkx as nx
import matplotlib.pyplot as plt
import time
from pathlib import Path

# Import our TSP solver modules
from src.graph import load_default_graph
from src.solvers import (
    nearest_neighbor_tsp,
    nearest_neighbor_multi_start,
    nearest_neighbor_with_2opt,
    improve_tour_2opt
)
from src.utils import calculate_tour_cost, validate_tour

print("✓ All imports successful")

In [ ]:
# Load the Singapore MRT/LRT network
print("Loading Singapore MRT/LRT network...")
G = load_default_graph()

print(f"\nNetwork Statistics:")
print(f"  Total stations: {G.number_of_nodes()}")
print(f"  Total connections: {G.number_of_edges()}")
print(f"  Network is connected: {nx.is_connected(G)}")
print(f"  Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")

<a id='us-301'></a>
## 2. US-301: Nearest Neighbor Heuristic

The Nearest Neighbor algorithm is a greedy constructive heuristic that:
1. Starts at a given station
2. Repeatedly visits the nearest unvisited station
3. Returns to the starting station

### 2.1 Single Run

In [ ]:
# Get a random starting station
start_station = list(G.nodes())[0]
print(f"Starting station: {start_station} - {G.nodes[start_station]['name']}")

# Run Nearest Neighbor
start_time = time.time()
nn_tour, nn_cost = nearest_neighbor_tsp(G, start_station=start_station)
nn_time = time.time() - start_time

print(f"\nNearest Neighbor Results:")
print(f"  Tour length: {len(nn_tour)} stations")
print(f"  Total travel time: {nn_cost:.2f} minutes ({nn_cost/60:.2f} hours)")
print(f"  Computation time: {nn_time:.4f} seconds")
print(f"\nFirst 10 stations in tour:")
for i, station_id in enumerate(nn_tour[:10]):
    station_name = G.nodes[station_id]['name']
    print(f"  {i+1}. {station_id} - {station_name}")

### 2.2 Multi-Start Nearest Neighbor

Since Nearest Neighbor is greedy, different starting points can yield different results. Let's try multiple starting stations.

In [ ]:
# Try multiple starting points
print("Running multi-start Nearest Neighbor (10 different starting points)...")

start_time = time.time()
best_tour, best_cost, best_start = nearest_neighbor_multi_start(G, num_starts=10)
multi_time = time.time() - start_time

print(f"\nMulti-Start Results:")
print(f"  Best starting station: {best_start} - {G.nodes[best_start]['name']}")
print(f"  Best tour cost: {best_cost:.2f} minutes ({best_cost/60:.2f} hours)")
print(f"  Computation time: {multi_time:.4f} seconds")
print(f"  Improvement over single run: {nn_cost - best_cost:.2f} minutes ({((nn_cost - best_cost)/nn_cost)*100:.2f}%)")

### 2.3 Comparing Different Starting Points

In [ ]:
# Test several specific starting stations
test_stations = list(G.nodes())[:20]  # First 20 stations
results = []

print("Testing 20 different starting stations...\n")

for station in test_stations:
    tour, cost = nearest_neighbor_tsp(G, start_station=station)
    results.append((station, G.nodes[station]['name'], cost))

# Sort by cost
results.sort(key=lambda x: x[2])

print("Top 5 best starting stations:")
for i, (station_id, station_name, cost) in enumerate(results[:5]):
    print(f"  {i+1}. {station_id} ({station_name}): {cost:.2f} min")

print("\nTop 5 worst starting stations:")
for i, (station_id, station_name, cost) in enumerate(results[-5:]):
    print(f"  {i+1}. {station_id} ({station_name}): {cost:.2f} min")

print(f"\nCost variation: {results[-1][2] - results[0][2]:.2f} minutes")

<a id='us-302'></a>
## 3. US-302: 2-Opt Local Search

The 2-Opt algorithm improves an existing tour by iteratively reversing segments. It continues until no further improvement can be found (reaching a local optimum).

### 3.1 Applying 2-Opt to Nearest Neighbor Solution

In [ ]:
# Use the Nearest Neighbor solution as initial tour
print("Initial tour (from Nearest Neighbor):")
print(f"  Cost: {nn_cost:.2f} minutes\n")

print("Applying 2-Opt optimization...")
start_time = time.time()
optimized_tour, original_cost, optimized_cost = improve_tour_2opt(
    nn_tour, 
    G,
    max_iterations=None,  # Run until convergence
    improvement_threshold=0.001,
    verbose=False
)
opt_time = time.time() - start_time

improvement = original_cost - optimized_cost
pct_improvement = (improvement / original_cost) * 100

print(f"\n2-Opt Optimization Results:")
print(f"  Original cost: {original_cost:.2f} minutes")
print(f"  Optimized cost: {optimized_cost:.2f} minutes")
print(f"  Improvement: {improvement:.2f} minutes ({pct_improvement:.2f}%)")
print(f"  Computation time: {opt_time:.4f} seconds")

### 3.2 Verbose 2-Opt Demonstration

Let's run 2-Opt with verbose output to see the optimization process.

In [ ]:
# Create a small test tour for demonstration
# Take first 10 nodes from NN tour
small_tour = nn_tour[:10]

print("Running 2-Opt with verbose output on a small tour (10 stations)...\n")
print("="*70)

# Create subgraph with only these nodes
small_nodes = set(small_tour)
small_G = G.subgraph(small_nodes).copy()

# Make it a complete subgraph for valid TSP
# (Add missing edges with high weight if needed)
for i, u in enumerate(small_tour):
    for v in small_tour[i+1:]:
        if not small_G.has_edge(u, v):
            # Use shortest path distance as weight
            try:
                path_length = nx.shortest_path_length(G, u, v, weight='weight')
                small_G.add_edge(u, v, weight=path_length)
            except nx.NetworkXNoPath:
                pass

if nx.is_connected(small_G):
    opt_small_tour, orig_small, opt_small = improve_tour_2opt(
        small_tour,
        small_G,
        max_iterations=10,
        verbose=True
    )
else:
    print("(Skipping verbose demo - subgraph not fully connected)")

print("="*70)

<a id='combined'></a>
## 4. Combined Workflow: Nearest Neighbor + 2-Opt

The recommended workflow is to use Nearest Neighbor to quickly generate an initial solution, then improve it with 2-Opt.

In [ ]:
# Use the combined function
print("Running combined Nearest Neighbor + 2-Opt workflow...\n")

start_time = time.time()
final_tour, initial_cost, final_cost = nearest_neighbor_with_2opt(
    G,
    start_station=best_start,  # Use the best starting station from multi-start
    verbose=True
)
total_time = time.time() - start_time

print(f"\n{'='*70}")
print(f"Total computation time: {total_time:.4f} seconds")

<a id='performance'></a>
## 5. Performance Analysis

Let's analyze the performance characteristics of both algorithms.

In [ ]:
# Summary table
print("\n" + "="*80)
print("PERFORMANCE SUMMARY")
print("="*80)

print(f"\nNetwork Size: {G.number_of_nodes()} stations, {G.number_of_edges()} connections")

print(f"\n{'Algorithm':<40} {'Time (s)':<12} {'Cost (min)':<15}")
print("-" * 80)
print(f"{'Nearest Neighbor (single start)':<40} {nn_time:<12.4f} {nn_cost:<15.2f}")
print(f"{'Nearest Neighbor (multi-start, n=10)':<40} {multi_time:<12.4f} {best_cost:<15.2f}")
print(f"{'2-Opt on NN solution':<40} {opt_time:<12.4f} {optimized_cost:<15.2f}")
print(f"{'Combined NN + 2-Opt':<40} {total_time:<12.4f} {final_cost:<15.2f}")

print(f"\nTotal tour distance: {final_cost:.2f} minutes = {final_cost/60:.2f} hours")
print(f"Average time per station: {final_cost/G.number_of_nodes():.2f} minutes")

### 5.1 Visualize Improvement Over Iterations

Let's track how 2-Opt improves the solution over multiple iterations.

In [ ]:
# Track improvement over time
from src.solvers.two_opt import improve_tour_2opt_fast

max_iter = 50
costs = [nn_cost]
current_tour = nn_tour.copy()

print(f"Tracking 2-Opt improvement over {max_iter} iterations...")

for i in range(max_iter):
    current_tour, _, current_cost = improve_tour_2opt(
        current_tour,
        G,
        max_iterations=1,
        improvement_threshold=0.0
    )
    costs.append(current_cost)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(range(len(costs)), costs, marker='o', markersize=3, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Tour Cost (minutes)', fontsize=12)
plt.title('2-Opt Optimization Progress', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=costs[0], color='r', linestyle='--', label=f'Initial (NN): {costs[0]:.2f} min', alpha=0.7)
plt.axhline(y=costs[-1], color='g', linestyle='--', label=f'Final: {costs[-1]:.2f} min', alpha=0.7)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nCost reduced from {costs[0]:.2f} to {costs[-1]:.2f} minutes")
print(f"Total improvement: {costs[0] - costs[-1]:.2f} minutes ({((costs[0] - costs[-1])/costs[0])*100:.2f}%)")

<a id='conclusions'></a>
## 6. Conclusions

### US-301: Nearest Neighbor Heuristic

**Strengths:**
- ✓ Fast execution (< 1 second for 189 nodes)
- ✓ Simple to understand and implement
- ✓ Deterministic results
- ✓ Provides reasonable baseline solution

**Limitations:**
- ✗ Greedy approach doesn't guarantee optimal solution
- ✗ Solution quality depends on starting station
- ✗ No backtracking or correction of early mistakes

**Best Practices:**
- Use multi-start to find better initial solutions
- Combine with local search for improved results

---

### US-302: 2-Opt Local Search

**Strengths:**
- ✓ Consistently improves initial solutions
- ✓ Configurable stopping criteria
- ✓ Reasonable computation time for 189 nodes
- ✓ Guaranteed to not worsen the solution

**Limitations:**
- ✗ Finds local optimum, not global optimum
- ✗ Quality depends on initial solution
- ✗ Can be slow on very large instances

**Best Practices:**
- Start with a good initial solution (e.g., from Nearest Neighbor)
- Use iteration limits for time-constrained scenarios
- Consider as pre-processor for more sophisticated algorithms

---

### Combined Workflow

The combination of **Nearest Neighbor + 2-Opt** provides an excellent balance of:
- Fast initial solution generation
- Systematic improvement through local search
- Reasonable computation time (< 30 seconds)
- High-quality results suitable for practical applications

This workflow is recommended for generating initial solutions that can be further refined with more advanced metaheuristics (Simulated Annealing, Genetic Algorithms, etc.) in future user stories.

## Next Steps

Future user stories will implement:
- **US-303**: Simulated Annealing (escape local optima)
- **US-304**: Genetic Algorithm (population-based search)
- **US-40X**: Visualization of tours on Singapore map
- **US-50X**: Web interface for interactive tour exploration